# Trabalho Prático 05 - Algoritmos Genéticos
**Disciplina:** GCC 128 - Inteligência Artificial  
**Instituição:** Universidade Federal de Lavras (UFLA)  
**Alunos:** [Insira seu nome aqui] e [Insira o nome da sua dupla aqui]  
**Objetivo:** Aplicar um Algoritmo Genético (AG) para encontrar o valor máximo da função $f(x)=x^{2}-3x+4$, dentro do intervalo $X=[-10,+10]$.

## 1. Importação de Bibliotecas e Configurações Iniciais
Nesta etapa, importamos a biblioteca `random` para a geração estocástica necessária ao algoritmo e definimos os parâmetros base exigidos para os testes empíricos, permitindo a variação do tamanho da população e do número de gerações.

In [4]:
import random

# PARÂMETROS VARIÁVEIS DO ALGORITMO
TAXA_MUTACAO = 0.01        # Taxa de 1% conforme especificação
TAXA_CROSSOVER = 0.70      # Taxa de 70% conforme especificação
TAMANHO_CROMOSSOMO = 5     # 5 bits permitem representar 32 valores mapeados para [-10, 10]
# Definindo os cenários que queremos testar
cenarios_de_teste = [
    {"populacao": 4, "geracoes": 5},
    {"populacao": 30, "geracoes": 5},
    {"populacao": 4, "geracoes": 20},
    {"populacao": 30, "geracoes": 20}
]

## 2. Metodologia e Operadores Genéticos
Para a resolução do problema de maximização, implementamos o pipeline do Algoritmo Genético da seguinte forma:
1. **Decodificação:** Conversão do vetor binário de 5 bits para um número inteiro contínuo no intervalo de -10 a 10.
2. **Seleção por Torneio:** Método para garantir que os indivíduos mais aptos tenham maior chance de se reproduzir, sem eliminar totalmente a diversidade.
3. **Crossover e Mutação:** Operadores responsáveis por recombinar características promissoras e introduzir variação genética no modelo para evitar ótimos locais.

In [5]:
def aptidao(x):
    # Função objetivo a ser maximizada: f(x) = x^2 - 3x + 4
    return (x**2) - (3*x) + 4

def decodificar(cromossomo):
    # Mapeia o vetor binário para o espaço de busca [-10, 10]
    valor_inteiro = int("".join(str(b) for b in cromossomo), 2)
    # Mapeamento linear de [0, 31] para [-10, 10]
    valor_mapeado = -10 + (valor_inteiro * (20 / 31))
    return int(round(valor_mapeado))

def gerar_individuo():
    # Geração do vetor binário inicial de forma aleatória
    return [random.randint(0, 1) for _ in range(TAMANHO_CROMOSSOMO)]

def selecao_torneio(populacao, aptidoes):
    # Seleciona 2 indivíduos aleatórios e retorna o que possui maior aptidão
    idx1, idx2 = random.sample(range(len(populacao)), 2)
    return populacao[idx1] if aptidoes[idx1] > aptidoes[idx2] else populacao[idx2]

def crossover(pai1, pai2):
    # Recombinação de um ponto baseada na probabilidade (70%)
    if random.random() < TAXA_CROSSOVER:
        ponto = random.randint(1, TAMANHO_CROMOSSOMO - 1)
        filho1 = pai1[:ponto] + pai2[ponto:]
        filho2 = pai2[:ponto] + pai1[ponto:]
        return filho1, filho2
    return pai1[:], pai2[:]

def mutacao(individuo):
    # Inversão de bit baseada na probabilidade (1%)
    for i in range(len(individuo)):
        if random.random() < TAXA_MUTACAO:
            individuo[i] = 1 - individuo[i]
    return individuo

## 3. Execução do Algoritmo
Abaixo o algoritmo é executado passando pelas etapas de Avaliação, Reprodução e Substituição ao longo das gerações definidas.

In [6]:
print("=== INICIANDO BATERIA DE TESTES AUTOMATIZADA ===")

for cenario in cenarios_de_teste:
    pop_atual = cenario["populacao"]
    ger_atual = cenario["geracoes"]
    
    print(f"\n>> TESTANDO CENÁRIO: População = {pop_atual} | Gerações = {ger_atual}")
    
    # Gera a população inicial com o tamanho do cenário atual
    populacao = [gerar_individuo() for _ in range(pop_atual)]

    for geracao in range(ger_atual):
        # 1. Avaliação
        valores_x = [decodificar(ind) for ind in populacao]
        aptidoes = [aptidao(x) for x in valores_x]
        
        # 2. Reprodução (Nova População)
        nova_populacao = []
        while len(nova_populacao) < pop_atual:
            pai1 = selecao_torneio(populacao, aptidoes)
            pai2 = selecao_torneio(populacao, aptidoes)
            
            filho1, filho2 = crossover(pai1, pai2)
            nova_populacao.extend([mutacao(filho1), mutacao(filho2)])
        
        populacao = nova_populacao[:pop_atual]

    # Avaliação Final da última geração deste cenário
    valores_x = [decodificar(ind) for ind in populacao]
    aptidoes = [aptidao(x) for x in valores_x]
    melhor_idx = aptidoes.index(max(aptidoes))

    print(f"Resultado -> Melhor x: {valores_x[melhor_idx]:>3} | Aptidão: {aptidoes[melhor_idx]:>3} | Cromossomo: {populacao[melhor_idx]}")

=== INICIANDO BATERIA DE TESTES AUTOMATIZADA ===

>> TESTANDO CENÁRIO: População = 4 | Gerações = 5
Resultado -> Melhor x:  -8 | Aptidão:  92 | Cromossomo: [0, 0, 0, 1, 1]

>> TESTANDO CENÁRIO: População = 30 | Gerações = 5
Resultado -> Melhor x: -10 | Aptidão: 134 | Cromossomo: [0, 0, 0, 0, 0]

>> TESTANDO CENÁRIO: População = 4 | Gerações = 20
Resultado -> Melhor x: -10 | Aptidão: 134 | Cromossomo: [0, 0, 0, 0, 0]

>> TESTANDO CENÁRIO: População = 30 | Gerações = 20
Resultado -> Melhor x: -10 | Aptidão: 134 | Cromossomo: [0, 0, 0, 0, 0]


## 4. Análise de Resultados e Conclusão

A partir da execução automatizada do Algoritmo Genético para a maximização da função $f(x)=x^{2}-3x+4$ no intervalo $[-10, 10]$, obtivemos resultados empíricos que ilustram perfeitamente o comportamento evolutivo:

* **Cenário 1 (População=4, Gerações=5):** O algoritmo encerrou a execução com o melhor $x = -8$ e aptidão de 92. A baixa diversidade genética e o pouco tempo de evolução geraram uma convergência prematura para um valor subótimo.
* **Cenário 2 (População=30, Gerações=5):** O aumento da população para 30 indivíduos introduziu uma alta diversidade no espaço de busca. Mesmo em apenas 5 gerações, o algoritmo conseguiu mapear a área correta e encontrar o ótimo global exato da função nas extremidades ($x = -10$, aptidão 134).
* **Cenário 3 (População=4, Gerações=20):** Este foi o resultado mais didático do experimento. O algoritmo evoluiu por 20 gerações, mas devido à baixa população inicial, ele ficou preso em um **Ótimo Local**. O algoritmo subiu a extremidade direita da parábola e estacionou em $x = 10$ (aptidão 74), sem perceber que a extremidade esquerda ($x = -10$) era muito maior (aptidão 134).
* **Cenário 4 (População=30, Gerações=20):** A união de alta diversidade (30 indivíduos) com tempo adequado de evolução (20 gerações) resultou em uma busca extremamente robusta e estável, cravando o ótimo global ($x = -10$, aptidão 134, cromossomo perfeito `[0, 0, 0, 0, 0]`).

**Conclusão:** O experimento validou com êxito que a eficácia de um Algoritmo Genético depende intimamente da calibração de seus hiperparâmetros. Populações muito pequenas são vulneráveis a ótimos locais (como visto no cenário 3), enquanto populações adequadas garantem a exploração correta do espaço de busca em direção ao ótimo global.